# 🚀 Deploy a Trained Policy with Rollout Recording

This notebook deploys a trained policy in a **loop** and records each rollout into a
[LeRobot v3.0](https://huggingface.co/docs/lerobot) dataset.

### Loop structure
```
for each rollout:
    1. Move home (not recorded)
    2. Deploy policy until Ctrl-C (recorded as one episode)
```

The recorded dataset contains the **raw** observations and actions (before action
translation), so it is directly compatible with the model's training format.

## 1. Configuration

Set your checkpoint path, robot address, recording output directory, and task name.

In [ ]:
import pathlib

# ----- Policy & Robot -----
CHECKPOINT_DIR = pathlib.Path("/data/models/your_policy_checkpoint")
SERVER_ENDPOINT = "robot_ip_address:50051"

# ----- Inference -----
INFERENCE_FREQUENCY_HZ: float = 10.0

# ----- Recording -----
RECORDING_OUTPUT_DIR = pathlib.Path("/data/rollout_recordings/recording_1")
TASK_NAME = "policy_rollout"  # human-readable task label stored in dataset
NUM_ROLLOUTS = 5             # number of deploy -> home cycles

# ----- Robot home -----
MOUNT_TYPE = "wall"  # "table" or "wall"

print(f"Checkpoint:   {CHECKPOINT_DIR}")
print(f"Robot:        {SERVER_ENDPOINT}")
print(f"Frequency:    {INFERENCE_FREQUENCY_HZ} Hz")
print(f"Output dir:   {RECORDING_OUTPUT_DIR}")
print(f"Rollouts:     {NUM_ROLLOUTS}")

## 2. Load the Policy

In [ ]:
import torch
from example_policies.robot_deploy.deploy_core.policy_manager import PolicyManager

device = "cuda" if torch.cuda.is_available() else "cpu"
policy_bundle = PolicyManager.load_single(CHECKPOINT_DIR, device)

print(f"✅ Policy loaded on {device}")

## 3. (Optional) Override Policy Attributes

In [ ]:
# Uncomment and modify as needed:
# policy_bundle.config.device = "cuda"
# policy_bundle.policy.to("cuda")
# policy_bundle.policy.config.n_action_steps = 8

## 4. Create the Recorder

The `RolloutRecorder` auto-detects observation state, action, and image features
from the loaded policy's metadata so the recorded dataset matches the model's
training format.

In [ ]:
import shutil
from pathlib import Path
from example_policies.robot_deploy.deploy_core.rollout_recorder import RolloutRecorder

output_path = Path(RECORDING_OUTPUT_DIR)
if output_path.exists():
    shutil.rmtree(output_path)
    print(f"Deleted existing dataset at {output_path}")

recorder = RolloutRecorder.from_policy_bundle(
    output_dir=RECORDING_OUTPUT_DIR,
    policy_bundle=policy_bundle,
    fps=int(INFERENCE_FREQUENCY_HZ),
    task_name=TASK_NAME,
)

## 5. Deploy + Record Loop

The loop below runs `NUM_ROLLOUTS` cycles of:
1. **Move home** (grippers open, consistent start pose — *not* recorded)
2. Wait for Enter to begin
3. **Deploy policy** until you press `Ctrl-C` — each step is recorded
4. **Rate the rollout**: press `s` (success) or `f` (failure) — stored in the dataset task label
5. Episode is saved

After all rollouts, the dataset is finalized and ready for training or upload.

⚠️ **Warning**: This will move the physical robot. Ensure a clear and safe workspace.

In [ ]:
from example_policies.robot_deploy.deploy import move_home
from example_policies.robot_deploy.deploy_core.deployment_structures import InferenceConfig
from example_policies.robot_deploy.deploy_core.inference_runner import InferenceRunner
from example_policies.robot_deploy.robot_io.connection import RobotConnection
from example_policies.robot_deploy.robot_io.robot_interface import RobotClient, RobotInterface

config = InferenceConfig(
    hz=INFERENCE_FREQUENCY_HZ,
    device=device,
    controller=RobotClient.CART_WAYPOINT,
)

with RobotConnection(SERVER_ENDPOINT) as stub:
    robot_interface = RobotInterface(stub, policy_bundle.config)
    runner = InferenceRunner(robot_interface, config, verbose=False)

    for rollout_idx in range(NUM_ROLLOUTS):
        # --- Move home (not recorded) ---
        print(f"\n{'=' * 60}")
        print(f"  Rollout {rollout_idx + 1}/{NUM_ROLLOUTS} — moving home")
        print(f"{'=' * 60}")
        move_home(robot_interface, mount=MOUNT_TYPE)

        input("Press Enter to start deployment (Ctrl-C to stop)...")

        # --- Deploy + record ---
        runner.step = 0  # reset step counter for each rollout
        recorder.start_episode()

        try:
            while True:
                result = runner.run_step_recorded(policy_bundle)
                recorder.record_step(result)
        except KeyboardInterrupt:
            print("\nRollout stopped.")

        # --- Rate the rollout ---
        while True:
            rating = input("Rate this rollout — (s)uccess or (f)ailure: ").strip().lower()
            if rating in ("s", "f"):
                break
            print("  Please enter 's' or 'f'.")

        outcome = "success" if rating == "s" else "failure"
        recorder.end_episode(outcome=outcome)

    # --- Final move home ---
    print(f"\n{'=' * 60}")
    print("  Moving home (final)")
    print(f"{'=' * 60}")
    move_home(robot_interface, mount=MOUNT_TYPE)

print(f"\n✅ All {NUM_ROLLOUTS} rollouts complete.")

## 6. Finalize Dataset

Consolidate the dataset (computes episode statistics, writes metadata).
After this step, `RECORDING_OUTPUT_DIR` contains a full LeRobot v3.0 dataset.

In [ ]:
recorder.close()

print(f"\n📁 Dataset saved to: {RECORDING_OUTPUT_DIR}")
print("You can load it with:")
print(f"  from lerobot.datasets.lerobot_dataset import LeRobotDataset")
print(f"  ds = LeRobotDataset(repo_id='local_only', root='{RECORDING_OUTPUT_DIR}')")

## 7. (Optional) Upload Dataset to Hugging Face

Upload the recorded rollout dataset to the Hugging Face Hub for sharing or remote training.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi, login

# --- Hugging Face Upload Configuration ---
HF_REPO_BASE = "pokeandwiggle/datasetname"  # e.g., "my-org/policy-rollouts"
PRIVATE = True
USE_LARGE_FOLDER_UPLOAD = True

# --- Append success rate to repo name ---
rate_pct = int(recorder.success_rate * 100)
n_success = sum(1 for o in recorder.outcomes if o == "success")
n_total = len(recorder.outcomes)
HF_REPO_ID = f"{HF_REPO_BASE}_{rate_pct}pct_{n_success}of{n_total}"
print(f"Success rate: {n_success}/{n_total} ({rate_pct}%)")
print(f"Repo ID:      {HF_REPO_ID}")

# --- Ensure we're logged in (uncomment if needed) ---
# login()

# --- Create repo if it doesn't exist ---
api = HfApi()
try:
    api.repo_info(repo_id=HF_REPO_ID, repo_type="dataset")
    print(f"✓ Repository {HF_REPO_ID} already exists")
except Exception:
    print(f"Creating new dataset repository: {HF_REPO_ID}")
    api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", private=PRIVATE)
    print(f"✓ Created repository: {HF_REPO_ID}")

# --- Load and upload the dataset ---
print(f"\nLoading dataset from {RECORDING_OUTPUT_DIR}...")
dataset = LeRobotDataset(repo_id=HF_REPO_ID, root=RECORDING_OUTPUT_DIR)

print(f"Uploading to Hugging Face Hub: {HF_REPO_ID}")
print(f"  Private: {PRIVATE}")
print(f"  Large folder upload: {USE_LARGE_FOLDER_UPLOAD}")
print(f"  Episodes: {dataset.meta.total_episodes}")
print(f"  Frames: {dataset.meta.total_frames}")
print()

dataset.push_to_hub(
    repo_id=HF_REPO_ID,
    private=PRIVATE,
    upload_large_folder=USE_LARGE_FOLDER_UPLOAD,
)

print(f"\n✅ Dataset uploaded successfully!")
print(f"   View at: https://huggingface.co/datasets/{HF_REPO_ID}")